# import


In [2]:
import pandas as pd
from pathlib import Path
import tensorflow as tf
from transformers import BertTokenizerFast, TFBertModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

SEED = 42
tf.random.set_seed(SEED)

I0000 00:00:1780662775.297446    5984 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780662775.326982    5984 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780662776.209017    5984 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


# gpu


In [3]:
gpus = tf.config.list_physical_devices("GPU")
print(gpus)

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# data load


In [4]:
parquet_path = Path("final_data.parquet")

df = pd.read_parquet(parquet_path)

print(df.shape)
print(df['fake'].value_counts().to_dict())
df.head()

(70000, 7)
{1: 35000, 0: 35000}


,review_text,fake,basic_linguistic_list,readability_list,sentiment_list,behavioral_list,clean_text
0,Food was great. Â Beer was great. Â Service wa...,1,"[39.0, 26.0, 6.0, 141.0, 110.0, 3.0, 17.0, 3.0...","[7.554174236665414, 74.65700000000002, 4.13800...","[0.08333333333333333, 0.6, 0.0, 0.0, 0.0, 0.0,...","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.0, 0.0]",food was great. beer was great. service was re...
1,"Nice place, good service and really good food....",1,"[18.0, 14.0, 2.0, 79.0, 60.0, 0.0, 10.0, 4.0, ...","[3.1291, 89.6067307692308, 2.375769230769233, ...","[0.7666666666666666, 0.7333333333333334, 0.0, ...","[3.0, 206.0, 103.0, 130.10764773832474, 1.0986...",nice place good service and really good food t...
2,This place is whiskey lovers' heaven. Most of ...,1,"[48.0, 37.0, 5.0, 204.0, 159.0, 2.0, 28.0, 13....","[6.742157984588678, 90.03152631578949, 2.58978...","[0.2, 0.48, 0.0, 0.0, 0.0, 0.0, 0.0]","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 0.0]",this place is whiskey lovers heaven. most of t...
3,If you ever find your way here ask for Katrina...,0,"[27.0, 23.0, 2.0, 116.0, 90.0, 1.0, 20.0, 4.0,...","[7.168621630094336, 95.84945652173916, 2.74717...","[0.9, 0.6375, 0.0, 0.0, 1.0, 0.0, 0.0]","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 0.0]",if you ever find your way here ask for katrina...
4,It's been a while since I've been impressed wi...,1,"[247.0, 175.0, 13.0, 960.0, 757.0, 20.0, 131.0...","[10.38987379855936, 73.02942641173854, 6.37919...","[0.4713541666666667, 0.6635416666666666, 0.0, ...","[8.0, 765.0, 109.28571428571429, 133.277547853...",it is been a while since i have been impressed...


# feature list


In [5]:
basic_cols = [f'basic_{i}' for i in range(13)]
read_cols  = [f'read_{i}' for i in range(6)]
senti_cols = [f'senti_{i}' for i in range(7)]
behav_cols = [f'behav_{i}' for i in range(9)]

df[basic_cols] = pd.DataFrame(df['basic_linguistic_list'].tolist(), index=df.index)
df[read_cols]  = pd.DataFrame(df['readability_list'].tolist(), index=df.index)
df[senti_cols] = pd.DataFrame(df['sentiment_list'].tolist(), index=df.index)
df[behav_cols] = pd.DataFrame(df['behavioral_list'].tolist(), index=df.index)

text_col = 'clean_text'
feat_cols = basic_cols + read_cols + senti_cols + behav_cols

print(df[feat_cols].shape)

(70000, 35)


# split


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=['fake']),
    df['fake'].astype('float32'),
    test_size=0.2,
    stratify=df['fake'],
    random_state=SEED
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.125,
    stratify=y_train,
    random_state=SEED
)

print(len(X_train), len(X_val), len(X_test))

49000 7000 14000


# hand-crafted feature scale


In [7]:
scaler = StandardScaler()

X_train_feat = scaler.fit_transform(X_train[feat_cols]).astype("float32")
X_val_feat = scaler.transform(X_val[feat_cols]).astype("float32")
X_test_feat = scaler.transform(X_test[feat_cols]).astype("float32")

print(pd.DataFrame(X_train_feat, columns=feat_cols).describe().loc[['mean', 'std']].T.head())

                 mean      std
basic_0  9.342116e-10  1.00001
basic_1  0.000000e+00  1.00001
basic_2 -3.736846e-09  1.00001
basic_3 -4.671058e-10  1.00001
basic_4  3.114038e-10  1.00001


# tokenize


In [8]:
MAX_LEN = 256

tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

train_enc = tokenizer(
    list(X_train["review_text"]),
    padding='max_length',
    truncation=True,
    max_length=MAX_LEN,
    return_tensors="np"
)

val_enc = tokenizer(
    list(X_val["review_text"]),
    padding="max_length",
    truncation=True,
    max_length=MAX_LEN,
    return_tensors="np"
)

test_enc = tokenizer(
    list(X_test["review_text"]),
    padding='max_length',
    truncation=True,
    max_length=MAX_LEN,
    return_tensors="np"
)

train_inputs = {
    "input_ids": train_enc["input_ids"],
    "attention_mask": train_enc["attention_mask"],
    "features": X_train_feat
}

val_inputs = {
    "input_ids": val_enc["input_ids"],
    "attention_mask": val_enc["attention_mask"],
    "features": X_val_feat
}

test_inputs = {
    "input_ids": test_enc["input_ids"],
    "attention_mask": test_enc["attention_mask"],
    "features": X_test_feat
}

# model


In [9]:
bert = TFBertModel.from_pretrained('bert-base-uncased')
bert.trainable = False

ids = tf.keras.Input((MAX_LEN,), dtype=tf.int32)
mask = tf.keras.Input((MAX_LEN,), dtype=tf.int32)
features = tf.keras.Input((len(feat_cols),), dtype=tf.float32)


# =========================
# Text branch
# =========================

text = bert(ids, attention_mask=mask).last_hidden_state
text = tf.keras.layers.Dense(128, activation='gelu')(text)

def masked_mean_pooling(inputs):
    x, mask = inputs
    mask = tf.cast(mask, tf.float32)
    mask = tf.expand_dims(mask, axis=-1)
    denom = tf.maximum(tf.reduce_sum(mask, axis=1), 1e-6)
    return tf.reduce_sum(x * mask, axis=1) / denom

text_pool = tf.keras.layers.Lambda(masked_mean_pooling)([text, mask])


# =========================
# Feature split
# =========================

ling_len = len(basic_cols) + len(read_cols) + len(senti_cols)

feat_linguistic = tf.keras.layers.Lambda(
    lambda x: x[:, :ling_len]
)(features)

feat_behavioral = tf.keras.layers.Lambda(
    lambda x: x[:, ling_len:]
)(features)


# =========================
# Linguistic feature branch
# =========================

linguistic_vec = tf.keras.layers.Dense(128, activation='gelu')(feat_linguistic)
linguistic_vec = tf.keras.layers.Dense(128, activation='gelu')(linguistic_vec)


# =========================
# Behavioral feature branch
# =========================

behavioral_vec = tf.keras.layers.Dense(128, activation='gelu')(feat_behavioral)
behavioral_vec = tf.keras.layers.Dense(128, activation='gelu')(behavioral_vec)


# =========================
# Feature fusion
# =========================

feature_vec = tf.keras.layers.Concatenate()([
    linguistic_vec,
    behavioral_vec
])

feature_vec = tf.keras.layers.Dense(128, activation='gelu')(feature_vec)


# =========================
# Fusion - Gated Fusion
# =========================

gate_input = tf.keras.layers.Concatenate()([text_pool, feature_vec])

gate = tf.keras.layers.Dense(128, activation='sigmoid')(gate_input)

x = tf.keras.layers.Multiply()([gate, text_pool])

inv_gate = tf.keras.layers.Lambda(lambda g: 1.0 - g)(gate)
y = tf.keras.layers.Multiply()([inv_gate, feature_vec])

x = tf.keras.layers.Add()([x, y])

x = tf.keras.layers.Dense(256, activation='gelu')(x)
x = tf.keras.layers.Dense(64, activation='gelu')(x)

out = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(
    inputs={
        'input_ids': ids,
        'attention_mask': mask,
        'features': features
    },
    outputs=out
)

model.summary()

I0000 00:00:1780662783.499483    5984 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 19829 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a Ber

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_3 (InputLayer)        [(None, 35)]                 0         []                            
                                                                                                  
 lambda_1 (Lambda)           (None, 26)                   0         ['input_3[0][0]']             
                                                                                                  
 lambda_2 (Lambda)           (None, 9)                    0         ['input_3[0][0]']             
                                                                                                  
 input_1 (InputLayer)        [(None, 256)]                0         []                            
                                                                                              

# compile


In [10]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# early stopping


In [11]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# train


In [12]:
history = model.fit(
    train_inputs,
    y_train,
    validation_data=(val_inputs, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/20


I0000 00:00:1780662791.517708    9006 service.cc:153] XLA service 0x72b247f9bbf0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780662791.517727    9006 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4090, Compute Capability 8.9 (Driver: 12.4.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.23.0)
I0000 00:00:1780662791.521373    9006 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1780662791.530764    9006 cuda_dnn.cc:461] Loaded cuDNN version 92300
I0000 00:00:1780662791.569204    9006 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1532/1532 [==============================] - 145s 91ms/step - loss: 0.5265 - accuracy: 0.7380 - val_loss: 0.4948 - val_accuracy: 0.7556
Epoch 2/20
1532/1532 [==============================] - 138s 90ms/step - loss: 0.5025 - accuracy: 0.7530 - val_loss: 0.4865 - val_accuracy: 0.7649
Epoch 3/20
1532/1532 [==============================] - 138s 90ms/step - loss: 0.4957 - accuracy: 0.7571 - val_loss: 0.4855 - val_accuracy: 0.7643
Epoch 4/20
1532/1532 [==============================] - 138s 90ms/step - loss: 0.4917 - accuracy: 0.7578 - val_loss: 0.4827 - val_accuracy: 0.7666
Epoch 5/20
1532/1532 [==============================] - 138s 90ms/step - loss: 0.4879 - accuracy: 0.7599 - val_loss: 0.4853 - val_accuracy: 0.7647
Epoch 6/20
1532/1532 [==============================] - 138s 90ms/step - loss: 0.4847 - accuracy: 0.7614 - val_loss: 0.4827 - val_accuracy: 0.7657
Epoch 7/20
1532/1532 [==============================] - 138s 90ms/step - loss: 0.4822 - accuracy: 0.7615 - val_loss: 0.4791 - val

# test


In [13]:
y_prob = model.predict(
    test_inputs,
    batch_size=64
)
y_pred = (y_prob > 0.5).astype(int)

print(f"acc  : {accuracy_score(y_test, y_pred):.4f}")
print(f"prec : {precision_score(y_test, y_pred):.4f}")
print(f"rec  : {recall_score(y_test, y_pred):.4f}")
print(f"f1   : {f1_score(y_test, y_pred):.4f}")

219/219 [==============================] - 32s 144ms/step
acc  : 0.7604
prec : 0.7149
rec  : 0.8660
f1   : 0.7833
